In [11]:
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
import pandas as pd
from sklearn.model_selection import train_test_split

seed = 14

df_original = pd.read_csv("./data_set.csv", sep="\t")


train, test = train_test_split(df_original, test_size=0.2, random_state=seed)

test.reset_index(inplace=True)
test.drop(columns=["index"], inplace=True)

def read_data(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    data = [line.strip() for line in lines]
    return data


df = pd.DataFrame.from_dict({'generated_text': read_data("./testes/preds_llama.txt"),
'target_text': test['estilo_autoral'].tolist(), "author": test['user_id'].tolist()})



In [12]:
test

,reescrito,estilo_academico,user_id,estilo_autoral
0,"<think>\nOkay, let's tackle this. The user wan...","Perdoe-me, mas a afirmação de que os membros d...",r2_bo_208,Oi? Petista não sabe nem falar. Eu hein
1,"<think>\nOkay, let's tackle this text rewritin...","Com profundo respeito e reconhecimento, dedico...",r2_lu_876,"Em homenagem ao meu Pastor Homem de D""us um t..."
2,"<think>\nOkay, let's tackle this rephrasing ta...",O profissional de jornalismo Felipe não é cons...,r2_bo_208,Não tenho como referência o jornalista Felipe.
3,"<think>\nOkay, let's tackle this rewriting tas...",O evento demonstrou uma situação problemática....,r2_cl_2165,O TIRO SAIU PELA CULATRA!!!... QUEM É BANDIDO...
4,"<think>\nOkay, let's tackle this reescrita tex...",Este indivíduo é claramente desagradável.,r2_gl_1071,Esse freixo é um FDP!
...,...,...,...,...
100,"<think>\nOkay, I need to rephrase the given te...",A deputada federal Janaína Paschoal manifestou...,r2_cl_2165,Janaína Paschoal questiona se CPI das ‘Fake N...
101,"<think>\nOkay, let's tackle this. The user wan...","Rindo de forma exagerada, manifeste a saída da...",r2_bo_344,AAHAHAHHAHHAHH VAZAAAAA
102,"<think>\nOkay, let's tackle this. The user wan...","Durante transmissão ao vivo, o ministro Abraha...",r2_cl_2165,AO VIVO: MINISTRO ABRAHAM WEINTRAUB CONFRONTA...
103,"<think>\nOkay, let's tackle this text rewritin...",Síria: Forças militares ocidentais encontram-s...,r2_lu_876,Síria: Forças militares do Ocidente já ocupam...


In [21]:

def compute_acc(candidates):

    model     = AutoModelForSequenceClassification.from_pretrained("../dataset/data/ustance/classification/results/checkpoint-520")
    tokenizer = AutoTokenizer.from_pretrained("pablocosta/bertabaporu-base-uncased")
    tokenizer.padding_side = "right"  # Fix weird overflow issue with fp16 training
    model = TextClassificationPipeline(model=model, tokenizer=tokenizer, top_k=1)
    preds = []
    for pred in model(candidates, truncation=True, padding=True, max_length=512):
        preds.append(int(pred[0]["label"].split("LABEL_")[1]))
    
    return preds

df["preds"] = compute_acc(df["generated_text"].tolist())

Device set to use cpu
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
OpenBLAS Warning : Detect OpenMP Loop and this application may hang. Please rebuild the library with USE_OPENMP=1 option.
Op

In [22]:
def get_real_preds(ids):
    labels = {'r2_lu_876': 0, 'r2_cl_2165': 1, 'r2_gl_1071': 2, 'r2_bo_344': 3, 'r2_bo_208': 4, 'r2_lu_94': 5}
    real_preds = [labels[text] for text in ids]
    return real_preds

In [23]:
df["real_preds"] = get_real_preds(test['user_id'].tolist())

In [24]:
df["preds"]

0      4
1      0
2      4
3      1
4      4
      ..
100    0
101    4
102    1
103    0
104    4
Name: preds, Length: 105, dtype: int64

In [26]:
df["iscorrect"] = (df["real_preds"] == df["preds"])

In [27]:
df["iscorrect"] = df["iscorrect"].apply(lambda x: "correct" if x else "incorrect")

In [28]:
df

,generated_text,target_text,author,preds,real_preds,iscorrect
0,"Desculpa, mas fala como se ele foi capaz de ex...",Oi? Petista não sabe nem falar. Eu hein,r2_bo_208,4,4,correct
1,Com respeito e admiração eu te cumpro ao pasto...,"Em homenagem ao meu Pastor Homem de D""us um t...",r2_lu_876,0,0,correct
2,Felipe jornalista não é um referencial,Não tenho como referência o jornalista Felipe.,r2_bo_208,4,4,correct
3,"Foi um abuso da nossa jornalista, ela forçou H...",O TIRO SAIU PELA CULATRA!!!... QUEM É BANDIDO...,r2_cl_2165,1,1,correct
4,esse hominídeio é desagradável,Esse freixo é um FDP!,r2_gl_1071,4,2,incorrect
...,...,...,...,...,...,...
100,Deputada federal Janaína Paschoal questiona CP...,Janaína Paschoal questiona se CPI das ‘Fake N...,r2_cl_2165,0,1,incorrect
101,Rindo barato sai o povo,AAHAHAHHAHHAHH VAZAAAAA,r2_bo_344,4,3,incorrect
102,"Ministro Weintraub, da Comissão Parlamentar Mi...",AO VIVO: MINISTRO ABRAHAM WEINTRAUB CONFRONTA...,r2_cl_2165,1,1,correct
103,Síria: Forças militares ocidentais se moviment...,Síria: Forças militares do Ocidente já ocupam...,r2_lu_876,0,0,correct


In [45]:
df_sample = df.groupby(['author', "iscorrect"]).sample(n=1)

In [46]:
df_sample

,generated_text,target_text,author,preds,real_preds,iscorrect
19,Trabalho de revisão bibliográfica? A essa altu...,O seu trabalho é uma revisão bibliográfica? v...,r2_bo_208,4,4,correct
99,Qual é a situação? Me afastou de possuir,O que aconteceu? Perdi,r2_bo_208,3,4,incorrect
21,eu e tu,Eu e tu,r2_bo_344,3,3,correct
36,Mandem a revolução p/ cair novamente,Rezando pra outra Revolução Pernambucana acon...,r2_bo_344,4,3,incorrect
7,Vejam o pronunciamento de Sérgio Moro sobre ST...,"ASSISTA: SERGIO MORO SE PRONUNCIA - STF, FACÇ...",r2_cl_2165,1,1,correct
98,Exclusivo com Bolsonaro - Parte 0 | [Nome da F...,ENTREVISTA EXCLUSIVA COM JAIR BOLSONARO - PAR...,r2_cl_2165,0,1,incorrect
55,"Acha melhor residência na Venezuela, sr. Wagne...","Ôh, Wagner Moura! Porque não vai morar na Ven...",r2_gl_1071,2,2,correct
95,Fazer um fogo no acampamento ou imitar os canh...,"Ou acender fogueira em acampamento, ou fazer ...",r2_gl_1071,4,2,incorrect
23,Assuntos Judaicos: Mulher brasileira serve na ...,Coisas Judaicas : Jovem brasileira na IDF,r2_lu_876,0,0,correct
76,O Povinho Judento!!! Quem você pensa que é um ...,Povo judeu em todo o mundo Que é um herói jud...,r2_lu_876,3,0,incorrect


In [47]:
from ollama import chat
from ollama import ChatResponse



In [48]:

prompt_original = """**Objective:** To generate clear and concise explanations for why a given pair of texts is classified as "correct" or "incorrect".

**Input:** You will receive text pairs as input. The text pair has a pre-assigned label ("correct" or "incorrect"). 

**Output:** Its task is to generate JSON output with the following keys: generated_text, target_text, label and explanation.

**Instructions:**

For each pair of texts, carefully analyze the generated and target texts and its relationship with the designated label **{assigned}**. Your explanation should be based on the following principles:

* **"correct" Label:** A pair of texts is labeled as “correct” if the generated text preserves syntactic, semantic and stylistic characteristics of the target text.
* **"incorrect" Label:** A pair of texts is labeled as “incorrect” if the generated text does not preserve syntactic, semantic and stylistic characteristics of the target text.

**Your generated explanation should:**

1. **Make direct reference to the content of the text.** Quote or paraphrase specific words or phrases that are indicative of the assigned label.
2. **Clearly state the reasoning.** Explicitly connect the textual evidence to the assigned label.
3. **Be concise and easy to understand.** Aim for a one or two-sentence explanation.
4. **Remain neutral and objective in your explanation.** Your role is to explain the stance, not agree or disagree with it.
5. **Return the explanation in Brazilian Portuguese.**

**Example of Expected Output:**
label: correct,
generated_text: "E o Anjo do Senhor apareceu-lhe e disse: Eis que conceberás e darás à luz um filho; e por causa do menino, chamarás o seu nome Isaías."
target_text: "E o Anjo do Senhor apareceu a esta mulher e disse-lhe: Eis que, agora, és estéril e nunca tens concebido; porém conceberás e terás um filho."
explanation: "A explicação deve ser aqui."

**Input:**
target_text: {target_text}
generated_text: {generated_text}
label: {assigned}
"""

In [49]:
def formatting_prompts_func(generated_text, target_text, label):
  return {'role': 'user', 'content': prompt_original.format(target_text=generated_text, generated_text=target_text, assigned=label)}

generated = []

for row in tqdm(df_sample.itertuples(), total=len(df_sample)):
  response: ChatResponse = chat(model='hf.co/unsloth/Qwen3-32B-GGUF:Q4_K_M', messages=[formatting_prompts_func(row.generated_text, row.target_text, row.iscorrect)], stream=False)
  generated.append(response.message.content)

100%|██████████| 11/11 [03:48<00:00, 20.78s/it]


In [50]:
df_sample["explanation"] = generated

In [51]:
df_sample

,generated_text,target_text,author,preds,real_preds,iscorrect,explanation
19,Trabalho de revisão bibliográfica? A essa altu...,O seu trabalho é uma revisão bibliográfica? v...,r2_bo_208,4,4,correct,"<think>\nOkay, let's tackle this. The user wan..."
99,Qual é a situação? Me afastou de possuir,O que aconteceu? Perdi,r2_bo_208,3,4,incorrect,"<think>\nOkay, I need to analyze why the given..."
21,eu e tu,Eu e tu,r2_bo_344,3,3,correct,"<think>\nOkay, let's see. The user provided a ..."
36,Mandem a revolução p/ cair novamente,Rezando pra outra Revolução Pernambucana acon...,r2_bo_344,4,3,incorrect,"<think>\nAlright, let me tackle this. The user..."
7,Vejam o pronunciamento de Sérgio Moro sobre ST...,"ASSISTA: SERGIO MORO SE PRONUNCIA - STF, FACÇ...",r2_cl_2165,1,1,correct,"<think>\nOkay, let's tackle this. The user wan..."
98,Exclusivo com Bolsonaro - Parte 0 | [Nome da F...,ENTREVISTA EXCLUSIVA COM JAIR BOLSONARO - PAR...,r2_cl_2165,0,1,incorrect,"<think>\nOkay, let's tackle this problem. The ..."
55,"Acha melhor residência na Venezuela, sr. Wagne...","Ôh, Wagner Moura! Porque não vai morar na Ven...",r2_gl_1071,2,2,correct,"<think>\nOkay, let's tackle this. The user wan..."
95,Fazer um fogo no acampamento ou imitar os canh...,"Ou acender fogueira em acampamento, ou fazer ...",r2_gl_1071,4,2,incorrect,"<think>\nAlright, let's tackle this problem. T..."
23,Assuntos Judaicos: Mulher brasileira serve na ...,Coisas Judaicas : Jovem brasileira na IDF,r2_lu_876,0,0,correct,"<think>\nOkay, let's tackle this. The user wan..."
76,O Povinho Judento!!! Quem você pensa que é um ...,Povo judeu em todo o mundo Que é um herói jud...,r2_lu_876,3,0,incorrect,"<think>\nOkay, let's tackle this. The user wan..."


In [52]:
df_sample.to_csv("./explanation_author.csv", sep=";", index=False)

In [1]:
import pandas as pd

df_sample = pd.read_csv("./explanation_author.csv", sep=";")

In [17]:
df_sample

,,,,,,generated_text,target_text,author,preds,real_preds,iscorrect,explanation
"Trabalho de revisão bibliográfica? A essa altura, a análise já está comprovada? Qual é a motivação para essa investigação? Quero ler por deusos","O seu trabalho é uma revisão bibliográfica? vc pode sintetizar a explicação do tema confrontando sua evolução conforme os anos. Nas últimas duas linhas escreve o porquê do trabalho. Quero ler, hein.",r2_bo_208,4,4,correct,"<think>\nOkay, let's tackle this. The user wan...",NaN,NaN,NaN,NaN,NaN,NaN
Qual é a situação? Me afastou de possuir,O que aconteceu? Perdi,r2_bo_208,3,4,incorrect,"<think>\nOkay, I need to analyze why the given...",NaN,NaN,NaN,NaN,NaN,NaN
eu e tu,Eu e tu,r2_bo_344,3,3,correct,"<think>\nOkay, let's see. The user provided a ...",NaN,NaN,NaN,NaN,NaN,NaN
Mandem a revolução p/ cair novamente,Rezando pra outra Revolução Pernambucana acontecer!,r2_bo_344,4,3,incorrect,"<think>\nAlright, let me tackle this. The user...",NaN,NaN,NaN,NaN,NaN,NaN
"Vejam o pronunciamento de Sérgio Moro sobre STF, quadrilhões, realidade do sistema prisional e o que ele fez, por @","ASSISTA: SERGIO MORO SE PRONUNCIA - STF, FACÇÕES CRIMINOSAS, PRESÍDIOS, ... via @",r2_cl_2165,1,1,correct,"<think>\nOkay, let's tackle this. The user wan...",NaN,NaN,NaN,NaN,NaN,NaN
Exclusivo com Bolsonaro - Parte 0 | [Nome da Fonte],ENTREVISTA EXCLUSIVA COM JAIR BOLSONARO - PARTE 0 via @,r2_cl_2165,0,1,incorrect,"<think>\nOkay, let's tackle this problem. The ...",NaN,NaN,NaN,NaN,NaN,NaN
"Acha melhor residência na Venezuela, sr. Wagner Moura. Sabe que tatuagem da suástica foi feita pela mesma jovem? Sabe se ex-presidente Fernando Henrique Cardoso, conhecido publicamente, apoio o PT e estuda no esquema político? Chama Ciro Gomes, chamado de Rivotril, o democrata? Vive os EUA","Ôh, Wagner Moura! Porque não vai morar na Venezuela? Sabia que a ""suástica tatuada"" na barriga da moça, foi ela mesma quem fez? Por acaso não viu, o FDP do FHC apoiando o pt e o esquemão, no qual, ele também se encontra? Falar que Ciro ""Rivotril"" é democrata! Putz! Tá no USA, né!",r2_gl_1071,2,2,correct,"<think>\nOkay, let's tackle this. The user wan...",NaN,NaN,NaN,NaN,NaN,NaN
Fazer um fogo no acampamento ou imitar os canhotos e ficar com o que está lendo,"Ou acender fogueira em acampamento, ou fazer igual aos da canhota, ficar com o ku letrado!",r2_gl_1071,4,2,incorrect,"<think>\nAlright, let's tackle this problem. T...",NaN,NaN,NaN,NaN,NaN,NaN
Assuntos Judaicos: Mulher brasileira serve na Força de Defesa de Israel,Coisas Judaicas : Jovem brasileira na IDF,r2_lu_876,0,0,correct,"<think>\nOkay, let's tackle this. The user wan...",NaN,NaN,NaN,NaN,NaN,NaN
O Povinho Judento!!! Quem você pensa que é um herói judeu? A comunidade de fãs da rede de colaboração para o álbum temático com este...,Povo judeu em todo o mundo Que é um herói judeu? Estamos postando indicados dos nossos fãs para o álbum com este...,r2_lu_876,3,0,incorrect,"<think>\nOkay, let's tackle this. The user wan...",NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df_sample.iloc[10]["generated_text"]

'<think>\nAlright, let\'s tackle this. The user provided a target text and a generated text, and the label is "incorrect". I need to explain why they\'re labeled as such based on the given principles.;;;;;;\n;;;;;;\nFirst, I\'ll look at the target text. It\'s a critical statement about the PT (Brazilian political party) accusing them of using the pandemic for ideological purposes. The key points are the PT exploiting the pandemic, the leader\'s quote about the virus being a good thing, and the claim that they don\'t care about lives or human suffering, focusing only on controlling the people. The target text uses strong language like "ideologicamente questionável" and mentions the left\'s tendency to use tragic events for power.;;;;;;\n;;;;;;\nNow, the generated text. It says the pandemic was a "achado" (a find) for the PT, quoting the same phrase from Lula. It mentions the lack of concern for victims and using the tragedy politically as typical of "petismo". ;;;;;;\n;;;;;;\nComparing 